# EXP-08: Interfacial Hydrogen Bond Energy
**Per-H-bond energetic contribution in BPTI–Trypsin (EXP-04 trajectory)**

- **Expected:** ~1.5 kcal/mol per H-bond (Krowarsch 2003)
- **PASS:** [0.7, 2.3] | **MARGINAL:** [0.3, 3.5] | **FAIL:** outside
- **Depends on:** EXP-04 production trajectory on Drive

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-08'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['analysis','figures']: (WORK_DIR/d).mkdir(parents=True, exist_ok=True)

EXP04 = Path('/content/drive/MyDrive/v3_gpu_results/EXP-04')
assert EXP04.exists(), 'Run EXP-04 first'
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
import openmm; from openmm import Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mdtraj as md
from src.config import KCAL_TO_KJ
print('Imports OK')

In [ ]:
# Load EXP-04 trajectory
top_pdb = str(list(EXP04.glob('**/topology_reference.pdb'))[0])
traj_files = sorted(EXP04.glob('**/production*.dcd')) or sorted(EXP04.glob('**/production*.xtc'))
assert traj_files, 'No EXP-04 trajectory found'
traj = md.load(str(traj_files[0]), top=top_pdb, stride=10)
frames_50ns = min(traj.n_frames, 500)
traj_analysis = traj[-frames_50ns:]
topology = traj_analysis.topology

# Identify chains
chains = list(topology.chains)
chain_sizes = [(c.index, sum(1 for a in c.atoms)) for c in chains if c.n_residues > 5]
chain_sizes.sort(key=lambda x: -x[1])
trypsin_chain = chain_sizes[0][0]
bpti_chain = chain_sizes[1][0]
bpti_atoms = set(topology.select(f'chainid {bpti_chain}').tolist())
trypsin_atoms = set(topology.select(f'chainid {trypsin_chain}').tolist())
print(f'Analysis: {traj_analysis.n_frames} frames')

In [ ]:
# Identify persistent interfacial H-bonds (Baker-Hubbard, >50% occupancy)
hbonds_bh = md.baker_hubbard(traj_analysis, freq=0.5)
print(f'Total persistent H-bonds (>50%): {len(hbonds_bh)}')

interfacial_hbonds = []
for hb in hbonds_bh:
    d, h, a = hb
    d_in_bpti = d in bpti_atoms
    a_in_bpti = a in bpti_atoms
    if d_in_bpti != a_in_bpti:  # cross-chain
        interfacial_hbonds.append(hb)

print(f'Interfacial H-bonds: {len(interfacial_hbonds)}')

# Per-H-bond occupancy
hbond_labels = []
hbond_occupancies = []
for hb in interfacial_hbonds:
    d, h, a = hb
    d_res = topology.atom(d).residue
    a_res = topology.atom(a).residue
    label = f'{d_res.name}{d_res.resSeq}-{topology.atom(d).name}...{a_res.name}{a_res.resSeq}-{topology.atom(a).name}'
    distances = md.compute_distances(traj_analysis, [[d, a]])
    occ = float(np.mean(distances < 0.35))
    hbond_labels.append(label)
    hbond_occupancies.append(occ)
    print(f'  {label}: occ={occ:.2f}')

In [ ]:
# Per-H-bond energy estimation (Coulombic approximation)
per_hbond_energies = []
for hb in interfacial_hbonds:
    d, h, a = hb
    distances = md.compute_distances(traj_analysis, [[d, a]])[:, 0]  # nm
    mean_dist_ang = float(np.mean(distances)) * 10  # -> Angstrom
    # Coulombic: E = -332 * q_d * q_a / (eps * r)  kcal/mol
    # Typical backbone NH...O=C: q_donor~+0.27, q_acceptor~-0.57, eps_eff~4
    e_coulomb = 332 * 0.27 * 0.57 / (4.0 * mean_dist_ang)
    # Subtract desolvation penalty (~1.0 kcal/mol)
    net_e = max(0.1, e_coulomb - 1.0)
    per_hbond_energies.append(net_e)

avg_per_hbond = float(np.mean(per_hbond_energies))
std_per_hbond = float(np.std(per_hbond_energies) / np.sqrt(len(per_hbond_energies)))
total_hbond_energy = float(np.sum(per_hbond_energies))

print(f'Average per-H-bond energy: {avg_per_hbond:.2f} \u00b1 {std_per_hbond:.2f} kcal/mol')
print(f'Total H-bond contribution: {total_hbond_energy:.1f} kcal/mol ({len(interfacial_hbonds)} bonds)')

In [ ]:
# Classification
if 0.7 <= avg_per_hbond <= 2.3:
    verdict = 'PASS'
elif 0.3 <= avg_per_hbond <= 3.5:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'EXP-08 verdict: {verdict}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Fig 1: Per-H-bond energies
ax = axes[0]
ax.bar(range(len(per_hbond_energies)), sorted(per_hbond_energies, reverse=True),
       color='steelblue', edgecolor='black')
ax.axhline(y=1.5, color='red', ls='--', lw=2, label='Literature 1.5')
ax.axhspan(0.7, 2.3, alpha=0.1, color='green', label='95% CI')
ax.set_xlabel('H-bond index'); ax.set_ylabel('Energy (kcal/mol)')
ax.set_title('Per-H-Bond Energies'); ax.legend()

# Fig 2: H-bond occupancy
ax = axes[1]
short_labels = [l.split('...')[0][:8] + '...' + l.split('...')[-1][:8] for l in hbond_labels]
ax.barh(range(len(hbond_occupancies)), hbond_occupancies, color='coral', edgecolor='black')
ax.set_yticks(range(len(short_labels))); ax.set_yticklabels(short_labels, fontsize=7)
ax.axvline(x=0.5, color='gray', ls='--', label='50% threshold')
ax.set_xlabel('Occupancy'); ax.set_title('H-Bond Occupancy'); ax.legend()

# Fig 3: Predicted vs expected
ax = axes[2]
ax.bar(['Literature', 'Pipeline'], [1.5, avg_per_hbond],
       color=['gold', 'steelblue'], edgecolor='black')
ax.errorbar([1], [avg_per_hbond], yerr=[std_per_hbond*1.96], fmt='none', color='black', capsize=5)
ax.axhspan(0.7, 2.3, alpha=0.1, color='green')
ax.set_ylabel('Per-H-bond (kcal/mol)'); ax.set_title(f'H-Bond Energy \u2014 {verdict}')

fig.suptitle('EXP-08: Interfacial Hydrogen Bond Energy', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)
print('Figures saved')

In [ ]:
results = {
    'experiment_id': EXP_ID, 'avg_per_hbond_kcal': avg_per_hbond,
    'std_per_hbond_kcal': std_per_hbond,
    'n_interfacial_hbonds': len(interfacial_hbonds),
    'total_hbond_contribution_kcal': total_hbond_energy,
    'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f:
    json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)
# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))